# 04 · PINN dầm Euler: phiên bản Python tối giản

Notebook **tự chứa mã**, chạy trên Colab bằng CPU. Ta chỉ dùng: tensor, một hàm lấy đạo hàm, một mạng nhỏ, một vòng `for` huấn luyện. Mục tiêu là hiểu **mỗi dòng code tương ứng với ý toán học nào**; script dự án là tài liệu đọc thêm sau bài này.

**Bài toán chuẩn hóa:** $u^{(4)}(x)=1$ trên $(0,1)$, với $u(0)=u(1)=u''(0)=u''(1)=0$. Nghiệm đúng $u_*(x)=(x^4-2x^3+x)/24$ chỉ dùng để kiểm tra sau huấn luyện.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(7)       # để lần chạy dễ tái lập
torch.set_num_threads(1)   # CPU nhỏ là đủ
print("PyTorch", torch.__version__)

## 1. Tensor và đạo hàm tự động

Mỗi hàng là một tọa độ $x$. Vì muốn đạo hàm theo $x$, ta bật `requires_grad=True`. `torch.autograd.grad` tính đạo hàm; `create_graph=True` cho phép tiếp tục lấy đạo hàm bậc cao.

In [ ]:
x = torch.tensor([[0.25]], dtype=torch.float64, requires_grad=True)
y = x**4
dy = torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]
print("dy/dx từ PyTorch:", dy.item())
print("4*x**3 tính tay:", (4*x**3).item())

Ta dùng đúng **một** hàm nhỏ để khỏi chép lại câu lệnh autograd nhiều lần. Với mỗi lần gọi, `y` và `x` là tensor có cùng shape `(N,1)`; mỗi đầu ra chỉ phụ thuộc vào tọa độ cùng hàng.

In [ ]:
def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

p = x**4 - 2*x**3 + x
p1 = grad(p, x)
p2 = grad(p1, x)
p3 = grad(p2, x)
p4 = grad(p3, x)
print("Đạo hàm bậc bốn phải là 24:", p4.item())
assert abs(p4.item() - 24) < 1e-10

**Phân biệt hai việc:** `grad(u,x)` lấy đạo hàm nghiệm theo tọa độ để lập phương trình. `loss.backward()` phía dưới lấy đạo hàm loss theo **trọng số mạng** để cập nhật mạng. Đừng đổi hai vai trò đó.

## 2. Tạo mạng và các điểm tính loss

Mạng nhận một số $x$ và trả một số $u_\theta(x)$. `Tanh` trơn nên có đạo hàm bậc bốn; ReLU không thích hợp cho residual dạng mạnh của bài này. `x_c` là 32 điểm trong miền; `x_b` gồm hai đầu dầm. Shape của cả hai là `(N,1)`.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1)
).double()

n_points = 32
x_c = ((torch.arange(n_points, dtype=torch.float64) + 0.5) / n_points)
x_c = x_c.reshape(-1, 1).requires_grad_(True)  # lấy u'''' theo tọa độ
x_b = torch.tensor([[0.0], [1.0]], dtype=torch.float64, requires_grad=True)  # lấy u'' ở biên
print("x_c shape", x_c.shape, "x_b shape", x_b.shape)
print("model(x_b) shape", model(x_b).shape)

## 3. Từ phương trình đến loss

Ở mỗi điểm miền: $r=u_\theta^{(4)}-1$. Ở hai đầu: chuyển vị $u_\theta$ và mômen tỷ lệ với $u_\theta''$ phải bằng 0. Ta dùng

$$L_{\rm PDE}=\operatorname{mean}(r^2),\qquad
L_{\rm BC}=\operatorname{sum}(u_b^2+u_b''{}^2),\qquad
L=L_{\rm PDE}+10L_{\rm BC}.$$

Trong ô huấn luyện, bốn dòng `u1`–`u4` là bốn đạo hàm; hai dòng `loss_pde`, `loss_bc` ứng với đúng hai hạng trên.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_history = []

for step in range(300):
    optimizer.zero_grad()             # xóa gradient trọng số từ bước trước

    u = model(x_c)  # forward trên điểm miền
    u1 = grad(u, x_c)
    u2 = grad(u1, x_c)
    u3 = grad(u2, x_c)
    u4 = grad(u3, x_c)
    loss_pde = ((u4 - 1)**2).mean()  # MSE residual u''''-1

    u_b = model(x_b)
    u_b1 = grad(u_b, x_b)
    u_b2 = grad(u_b1, x_b)
    loss_bc = (u_b**2 + u_b2**2).sum()  # u=0 và u''=0 ở hai đầu

    loss = loss_pde + 10*loss_bc
    loss.backward()                   # gradient theo trọng số mạng
    optimizer.step()                  # cập nhật trọng số
    loss_history.append(loss.item())  # lấy số, không giữ đồ thị

print("loss đầu:", loss_history[0], "loss cuối:", loss_history[-1])

**Dừng và đọc code:** `model(x_c)` cho nghiệm ở đâu? Dòng nào kiểm tra PDE? Dòng nào kiểm tra bốn điều kiện biên? Vì sao cần cả `backward()` và `step()`? Loss giảm chưa đủ chứng tỏ nghiệm đúng trên toàn miền.

## 4. Kiểm tra trên điểm chưa dùng để huấn luyện

Tạo 101 điểm mới, so nghiệm mạng với $u_*$. Mômen chuẩn hóa là $M=-u''$; nghiệm đúng $M_*=x(1-x)/2$. Chuẩn sai số ở đây dùng tổng bình phương trên lưới đều, dễ tính và gần với chuẩn $L^2$.

In [ ]:
x_test = torch.linspace(0, 1, 101, dtype=torch.float64).reshape(-1, 1)
x_test.requires_grad_(True)  # verify vẫn cần đạo hàm theo x để lấy mômen
u_test = model(x_test)
u_exact = (x_test**4 - 2*x_test**3 + x_test) / 24

u_test1 = grad(u_test, x_test)
u_test2 = grad(u_test1, x_test)
M_test = -u_test2
M_exact = x_test*(1-x_test)/2

error_u = torch.sqrt(((u_test-u_exact)**2).sum() / (u_exact**2).sum())
error_M = torch.sqrt(((M_test-M_exact)**2).sum() / (M_exact**2).sum())
print("sai số chuyển vị:", error_u.item())
print("sai số mômen:", error_M.item())
print("u ở hai đầu:", model(x_b).detach().reshape(-1).tolist())  # detach trước khi chuyển sang list

## 5. Vẽ kết quả và đặt câu hỏi

`.detach()` tách tensor khỏi đồ thị autograd trước khi đưa số sang Matplotlib. Đồ thị cho biết nghiệm mạng khớp ở đâu, nhưng luôn đọc cùng sai số số học ở ô trước.

In [ ]:
xx = x_test.detach().reshape(-1).tolist()
plt.plot(xx, u_exact.detach().reshape(-1).tolist(), label="nghiệm đúng")
plt.plot(xx, u_test.detach().reshape(-1).tolist(), "--", label="PINN")
plt.xlabel("x"); plt.ylabel("u(x)")
plt.grid(); plt.legend(); plt.show()

## 6. Tự làm một thay đổi

Đổi **một** trong ba lựa chọn: `n_points = 64`, hệ số biên `10` thành `100`, hoặc số bước `300` thành `600`. Chọn **Runtime → Restart and run all** để chạy lại từ đầu; ghi loss, sai số chuyển vị, sai số mômen trước/sau thay đổi. Giữ seed bằng 7 khi so sánh lần đầu, sau đó thử thêm seed khác.

**Nộp:** notebook chạy được từ đầu, một bảng hai cấu hình và một đoạn giải thích kết quả. Đọc [trang bài 04](https://nhannguyenminh.github.io/intro2pinns/learn-python/04-torch-pinn.html) để đối chiếu các dòng code với công thức. Khi hiểu notebook ngắn này, mới chuyển sang [notebook tích hợp 05](05-beam-fem-to-pinn.ipynb) hoặc [script đầy đủ](04-torch-pinn.py).